# Fleche Storage Backends

This notebook demonstrates the usage of the different storage backends available in `fleche`.

## Memory Storage

The `Memory` storage backend keeps all the cached data in memory. This is the simplest backend and is useful for testing or when you don't need to persist the cache.

In [1]:
from fleche import fleche, cache, Cache
from fleche.storage import Memory
from fleche.metadata import PandasDB

memory_cache = Cache(PandasDB({}), storage=Memory({}))

with cache(memory_cache):
    @fleche
    def add(a, b):
        print(f"Executing add({a}, {b})")
        return a + b
    
    add(2, 3)
    add(2, 3)
print('Cache keys:', list(memory_cache.storage.list()))

Executing add(2, 3)
Cache keys: ['99c82173105386ba4280a7875e8981fa8f259f7f9596778f08f57a4968e367fe']


## CloudpickleFile Storage

The `CloudpickleFile` storage backend serializes the data using `cloudpickle` and stores it in files. You need to specify a root directory for the cache.

In [2]:
import shutil
from fleche import fleche, cache, Cache
from fleche.storage import CloudpickleFile
from fleche.metadata import PandasDB

shutil.rmtree('.cloudpickle_cache', ignore_errors=True)
cp_cache = Cache(PandasDB({}),  CloudpickleFile(root='.cloudpickle_cache'))

with cache(cp_cache):
    @fleche
    def mul(a, b):
        print(f"Executing mul({a}, {b})")
        return a * b
    mul(3, 4)
    mul(3, 4)
    print(*cache().storage.root.iterdir())

!ls .cloudpickle_cache

Executing mul(3, 4)
.cloudpickle_cache/2f7cd4d613ef2566d92b6211ec21360c09dbdc3a7daed720fa2662b720bb2ebb
2f7cd4d613ef2566d92b6211ec21360c09dbdc3a7daed720fa2662b720bb2ebb


## BagOfHoldingH5File Storage

The `BagOfHoldingH5File` storage backend uses the `bagofholding` library to store data in HDF5 files. This is a good option for storing large numerical arrays efficiently.

In [3]:
import shutil
from fleche import fleche, cache, Cache
from fleche.storage import BagOfHoldingH5File
from fleche.metadata import PandasDB

shutil.rmtree('.boh_cache', ignore_errors=True)
boh_cache = Cache(PandasDB({}), BagOfHoldingH5File(root='.boh_cache'))

with cache(boh_cache):
    @fleche
    def power(a, b):
        print(f"Executing power({a}, {b})")
        return a ** b

    power(2, 8)
    power(2, 8)
!ls .boh_cache

Executing power(2, 8)
8ccfc1f2c77e2e4b837e524bbc2e3171441b0400ced07886f7ba4fdfe417122a


# Clean Up

Remove cache dirs to ensure clean CI.

In [4]:
!rm -rf .cloudpickle_cache .boh_cache